<a href="https://colab.research.google.com/github/CSI5195-Project/Fraud-Detection/blob/main/ModelSNNPC_withProba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

danieldumi_snn_metrics_path = kagglehub.utility_script_install('danieldumi/snn-metrics')
evavivante_snn_metrics_2_path = kagglehub.utility_script_install('evavivante/snn-metrics-2')

print('Data source import complete.')


In [ ]:
# %% [code]
# %% [code]
# %% [code]
# imports
import os
os.system('pip install snntorch')

import numpy as np
import torch
import snntorch.functional as SF
import snntorch as snn
import torch.nn as nn

from snntorch.functional.acc import _prediction_check, _population_code
from snntorch.surrogate import fast_sigmoid
from torch import from_numpy
from torch.utils.data import DataLoader, Dataset
from snn_metrics_2 import evaluate, evaluate_business_constraint, evaluate_fairness

class DatasetBAF(Dataset):
    def __init__(self, x, y, indices):
        self.data = x
        self.targets = y
        self.indices = indices

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        return self.data[index], self.targets[index], self.indices[index]

class CSNNPC(nn.Module):
    def __init__(self, num_inputs, num_outputs, population,  betas, spike_grad, num_steps, thresholds):
        super().__init__()
        self.num_inputs = num_inputs
        self.num_outputs = num_outputs
        self.population = population
        self.betas = betas
        self.spike_grad = spike_grad
        self.num_steps = num_steps
        self.thresholds = thresholds
        self.architecture = f"Conv1d(1, 32, 2) + MaxPool(2) + LIF + Conv1d(32, 128, 2) + MaxPool(2) + LIF + Conv1d(128, 256, 2) + MaxPool(2) + LIF + Linear(768, {self.population}) + LIF"
        self.conv1 = nn.Conv1d(1, 32, 2)
        self.mp1 = nn.MaxPool1d(2)
        self.lif1 = snn.Leaky(beta=self.betas[0], spike_grad=spike_grad, threshold=self.thresholds[0], learn_beta=True, learn_threshold=True)
        self.conv2 = nn.Conv1d(32, 128, 2)
        self.mp2 = nn.MaxPool1d(2)
        self.lif2 = snn.Leaky(beta=self.betas[1], spike_grad=spike_grad, threshold=self.thresholds[1], learn_beta=True, learn_threshold=True)
        self.conv3 = nn.Conv1d(128, 256, 2)
        self.mp3 = nn.MaxPool1d(2)
        self.lif3 = snn.Leaky(beta=self.betas[2], spike_grad=spike_grad, threshold=self.thresholds[2], learn_beta=True, learn_threshold=True)
        self.fc1 = nn.Linear(768, self.population)
        self.lif4 = snn.Leaky(beta=self.betas[3], spike_grad=spike_grad, threshold=self.thresholds[3], learn_beta=True, learn_threshold=True, output=True)

    def forward(self, x):
        """
        Forward pass of the network.
        ------------------------------------------------------
        Args:
            x (torch.Tensor): input tensor
        ------------------------------------------------------
        Returns:
            cur_last_rec (torch.Tensor): tensor with the last current values
            spk_last_rec (torch.Tensor): tensor with the last spike values
            mem_last_rec (torch.Tensor): tensor with the last membrane values
        """
        cur_last_rec = []
        spk_last_rec = []
        mem_last_rec = []
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        for _ in range(self.num_steps):
            cur1 = self.mp1(self.conv1(x))
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.mp2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            cur3 = self.mp3(self.conv3(spk2))
            spk3, mem3 = self.lif3(cur3, mem3)
            cur4 = self.fc1(spk3.flatten(1))
            spk4, mem4 = self.lif4(cur4, mem4)
            cur_last_rec.append(cur4)
            spk_last_rec.append(spk4)
            mem_last_rec.append(mem4)
        return torch.stack(cur_last_rec, dim=0), torch.stack(spk_last_rec, dim=0), torch.stack(mem_last_rec, dim=0)

    def get_architecture(self):
        """
        Get the architecture of the network.
        ------------------------------------------------------
        Returns:
            architecture (str): string with the architecture
        """
        return self.architecture

    def get_parameters(self):
        """
        Get the parameters of the network.
        ------------------------------------------------------
        Returns:
            parameters (list): list with the parameters
        """
        return [p for p in self.parameters() if p.requires_grad]

    def get_num_params(self):
        """
        Get the number of parameters of the network.
        ------------------------------------------------------
        Returns:
            num_params (int): number of parameters
        """
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class ModelSNNPC(object):
    """
    Class to create a Spike Neural Network with Population Coding.
    ------------------------------------------------------
    Args:
        num_features (int): number of features
        num_classes (int): number of classes
        population (int): number of neurons in the population
        class_weights (tuple): weights for the classes
        batch_size (int): size of the batch
        betas (tuple): betas for the network
        slope (int): slope for the fast sigmoid
        thresholds (tuple): thresholds for the network
        num_epochs (int): number of epochs
        num_steps (int): number of steps
        adam_betas (tuple): betas for the Adam optimizer
        learning_rate (float): learning rate for the optimizer
        verbose (int): level of verbosity
    """
    def __init__(self, num_features, num_classes, population, class_weights=(0.02, 0.98), batch_size=128, betas=(0.5,0.5,0.5), slope=25, thresholds=(0.2,0.2,0.2), num_epochs=1, num_steps=20, adam_betas=(0.9, 0.999), learning_rate=1e-5, verbose=1):
        device = torch.device("cpu")
        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        self.num_features = num_features
        self.num_classes = num_classes
        self.population = population
        self.batch_size = batch_size
        self.num_epochs = num_epochs
        self.num_steps = num_steps
        self.adam_betas = adam_betas
        self.verbose = verbose
        self.betas = betas
        self.slope = slope
        self.thresholds = thresholds
        self.spike_grad = fast_sigmoid(slope=slope)
        self._dtype = torch.float
        self._device = device
        self.network = self._loadnetwork()
        self.class_weights = torch.tensor(class_weights, dtype=self._dtype, device=self._device)
        self._loss_fn = SF.ce_count_loss(weight=self.class_weights, population_code=True, num_classes=2)
        self._optimizer = torch.optim.Adam(self.network.parameters(), lr=learning_rate, betas=self.adam_betas)


    def _load_data(self, x, y, indices):
        """Load the data."""
        x_np = from_numpy(x.values).float().unsqueeze(1)
        y_np = from_numpy(y.values).int()
        ds = DatasetBAF(x_np, y_np, indices)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=False)
        return loader

    def _loadnetwork(self):
        """Load the network."""
        network = CSNNPC(self.num_features, self.num_classes, self.population, self.betas, self.spike_grad, self.num_steps, self.thresholds)
        print(network) if self.verbose >= 2 else None
        return network.to(self._device)

    def fit(self, x_train, y_train):
        """
        Fit the model to the training set.
        ------------------------------------------------------
        Args:
            x_train (pd.DataFrame): dataframe with the training features
            y_train (pd.Series): series with the training labels
        """
        indices = torch.arange(len(x_train))
        self._train_loader = self._load_data(x_train, y_train,indices)
        for epoch in range(self.num_epochs):
            print(f"Epoch - {epoch}") if self.verbose >= 2 else None
            train_batch = iter(self._train_loader)
            for data, targets, indices in train_batch:
                data = data.to(self._device)
                targets = targets.to(self._device, dtype=torch.long)
                self.network.train()
                _, spk_rec, _ = self.network(data)
                loss_val = self._loss_fn(spk_rec, targets)
                self._optimizer.zero_grad()
                loss_val.backward()
                self._optimizer.step()


    def predict(self, x_test, y_test=None):
        """
        Predict the labels of the test set.
        ------------------------------------------------------
        Args:
            x_test (pd.DataFrame): dataframe with the test features
            y_test (pd.Series): series with the test labels
        ------------------------------------------------------
        Returns:
            predictions (np.array): array with the predictions
            test_targets (np.array): array with the true values
        """
        if y_test is None:
            y_test = torch.zeros(len(x_test))  # dummy labels For SHAP part

        indices = torch.arange(len(x_test))
        self._test_loader = self._load_data(x_test, y_test, indices)
        predictions = []
        prediction_probs = []
        test_targets = []
        original_indices = []

        with torch.no_grad():
            self.network.eval()
            for data, targets, indices in iter(self._test_loader):
                data = data.to(self._device)
                targets = targets.to(self._device, dtype=torch.long)
                _, spk_rec, _ = self.network(data)
                _, _, num_outputs = _prediction_check(spk_rec)
                pop_out = _population_code(spk_rec, self.num_classes, num_outputs)
                _, idx = pop_out.max(1)
                probs = torch.nn.functional.softmax(pop_out, dim=1)
                # predictions = np.append(predictions, idx.cpu().numpy())
                # test_targets = np.append(test_targets, targets.cpu().numpy())
                predictions.extend(idx.cpu().numpy()) # add prediction to list
                test_targets.extend(targets.cpu().numpy()) # add original label to list
                prediction_probs.extend(probs.cpu().numpy())
                original_indices.extend(indices.cpu().numpy()) # add original index of example to list

        # sorting to match original order of examples
        index_to_prediction = {idx: pred for idx, pred in zip(original_indices, predictions)}
        index_to_target = {idx: pred for idx, pred in zip(original_indices, test_targets)}
        index_to_probs = {idx: prob for idx, prob in zip(original_indices, prediction_probs)}

        sorted_predictions = [index_to_prediction[idx] for idx in sorted(original_indices)]
        sorted_targets = [index_to_target[idx] for idx in sorted(original_indices)]
        sorted_probs = [index_to_probs[idx] for idx in sorted(original_indices)]

        return sorted_predictions, sorted_targets, sorted_probs


    def evaluate(self, targets, predicted):
        """Evaluate the model using the confusion matrix and some metrics.
        ------------------------------------------------------
        Args:
            targets (list): list of true values
            predicted (list): list of predicted values
        ------------------------------------------------------
        Returns:
            cm (np.array): confusion matrix
            tn (int): true negative
            fp (int): false positive
            fn (int): false negative
            tp (int): true positive
            accuracy (float): accuracy of the model
            precision (float): precision of the model
            recall (float): recall of the model
            fpr (float): false positive rate of the model
            tnr (float): true negative rate of the model
            f1_score (float): f1 score of the model
            auc (float): area under the curve of the model
        """
        metrics = evaluate(targets, predicted)
        return metrics

    def evaluate_business_constraint(self, y_test, predictions):
        """Evaluate the model using the business constraint of 5% FPR.
        ------------------------------------------------------
        Args:
            x_test (pd.DataFrame): dataframe with the test features
            y_test (pd.Series): series with the test labels
            predictions (np.array): array with the predictions
        ------------------------------------------------------
        Returns:
            threshold (float): threshold for the model
            fpr@5FPR (float): false positive rate of the model
            recall@5FPR (float): recall of the model
            tnr@5FPR (float): true negative rate of the model
            accuracy@5FPR (float): accuracy of the model
            precision@5FPR (float): precision of the model
            f1_score@5FPR (float): f1 score of the model
        """
        metrics = evaluate_business_constraint(y_test, predictions)
        return metrics

    def evaluate_fairness(self, x_test, y_test, predictions, sensitive_attribute, attribute_threshold):
        """Evaluate the model using the Aequitas library.
        ------------------------------------------------------
        Args:
            x_test (pd.DataFrame): dataframe with the test features
            y_test (pd.Series): series with the test labels
            predictions (np.array): array with the predictions
            sensitive_attribute (str): name of the sensitive attribute
            attribute_threshold (float): threshold for the sensitive attribute
        ------------------------------------------------------
        Returns:
            fpr_ratio (float): false positive rate ratio of the model
            fnr_ratio (float): false negative rate ratio of the model
        """
        metrics = evaluate_fairness(x_test, y_test, predictions, sensitive_attribute, attribute_threshold)
        return metrics

